# 01. 지식그래프 구축 (RDF 트리플 → Neo4j + n10s)
수업 노트북(Milvus 벡터 RAG)의 **'문서 → 청크 → 임베딩 → 벡터DB'** 대신 **'공지 → 추출 → 트리플 → 그래프DB'** 로 대체한 파이프라인입니다.
이 노트북은 **얇은 실행·점검용**입니다. 실제 로직은 `src/maplerag/` 에 있습니다.

| 수업 노트북 | 이 프로젝트 |
|---|---|
| 문서 로드·청킹 | `ingest/` (Open API 수집) + `extract/sections.py` (■ 소제목 단위 청크) |
| 임베딩 + Milvus 적재 | `extract/normalizer.py` (트리플화·정규화) + `graph/load_n10s.py` (n10s 적재) |
| 데이터 품질 확인 | `graph/validate.py` (SHACL 위반 건수) |

**비용 안전장치**: 아래 `RUN_LLM` 은 기본 `False` 입니다. 켜면 API 호출·비용이 발생합니다.

In [ ]:
RUN_LLM = False       # True: LLM 추출 일괄 실행(유료). 기본은 캐시된 결과를 사용
RUN_LOAD = True       # Neo4j 적재(멱등: 다시 실행해도 개수 불변)
import json
from collections import Counter
from maplerag.config import key_status, settings
from maplerag.graph.neo4j_client import check_connection
print("키 상태(값은 출력 안 함):", key_status(settings))
print("Neo4j:", check_connection())

## 1. 수집 결과 확인 (RawDocument)
공식 Open API 의 최신 공지만 사용합니다(robots.txt 허용 범위, 비공식 출처 미수집). 재수집은 `maplerag.ingest.run_notices` (일 1,000회 한도).

In [ ]:
from maplerag.ingest.base import RAW_DIR
docs = [json.loads(l) for l in (RAW_DIR / "docs" / "notices.jsonl").read_text(encoding="utf-8").splitlines()]
print("공지", len(docs), "건 | 유형:", dict(Counter(d["doc_type"] for d in docs)), "| tier:", dict(Counter(d["tier"] for d in docs)))
dates = [d["meta"]["date"][:10] for d in docs if d["doc_type"] == "patch_note"]
print("패치노트 기간:", min(dates), "~", max(dates))

## 2. LLM 추출 결과 확인
구조화 출력(Pydantic) → 코드 검증(근거 문장이 본문에 있는지, 값이 근거에 있는지)을 거친 결과입니다. 실행은 `maplerag.extract.run_batch` (유료).

In [ ]:
if RUN_LLM:
    from maplerag.extract import run_batch
    run_batch.main()
rows = [json.loads(l) for l in (RAW_DIR / "docs" / "extraction_all.jsonl").read_text(encoding="utf-8").splitlines()]
ok = [r for r in rows if not r["error"]]
print(f"청크 {len(rows)}개 중 성공 {len(ok)} / 실패 {len(rows)-len(ok)}  (실패는 API 크레딧 부족 때문)")
print("변경", sum(len(r["changes"]) for r in ok), "건 / 관계", sum(len(r["relations"]) for r in ok), "건 / 코드 검증 폐기", sum(len(r["dropped"]) for r in ok), "건")
print("변경 종류:", dict(Counter(c["change_type"] for r in ok for c in r["changes"])))

## 3. 정규화 → RDF → SHACL 검증
별칭 병합·중복 제거·시간(validFrom/inPatch)·출처(prov:wasDerivedFrom) 부여. 같은 입력을 두 번 넣어도 같은 결과(멱등)여야 합니다.

In [ ]:
from maplerag.extract.run_normalize import RDF_DIR, build
from maplerag.graph.validate import load, validate_graph
g, norm = build()
g2, _ = build()
print("트리플", len(g), "| 멱등:", set(g) == set(g2))
full = load(RDF_DIR / "notices.ttl")
for t in g:
    full.add(t)
rep = validate_graph(full)
print("SHACL conforms =", rep["conforms"], "| 위반", rep["violation_count"], "건", rep["by_path"])
print("정규화 통계:", dict(norm.stats))

## 4. Neo4j(n10s) 적재
멱등 적재입니다. 기존 그래프를 지우고 다시 넣으려면 터미널에서 `uv run python -m maplerag.graph.load_n10s --drop-old` (확인 프롬프트 있음).

In [ ]:
from maplerag.graph import load_n10s as L
if RUN_LOAD:
    with L.get_driver() as d, d.session() as s:
        print(L.ensure_config(s))
        print(L.import_graph(s, L.read_graph()))
        print("그래프 건수:", L.counts(s))

## 5. 연결 관계 둘러보기 (Cypher)
Neo4j Browser(`http://localhost:7475`)에서 같은 쿼리를 실행하면 그래프로 볼 수 있습니다.

In [ ]:
from maplerag.rag.pipeline import run_read_query
print(run_read_query("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC LIMIT 12"))
print(run_read_query('''MATCH (j:maple__Job)-[:maple__hasSkill]->(s:maple__Skill)<-[:maple__changes]-(c:maple__PatchChange)-[:maple__inPatch]->(p:maple__Patch)
RETURN j.rdfs__label AS job, s.rdfs__label AS skill, c.maple__beforeValue AS before, c.maple__afterValue AS after, toString(p.maple__patchDate) AS date LIMIT 5'''))